
# GPU Roofline & Kernel‑Parameter Study

This notebook automates three experiments on **your OpenCL kernel**:

1. **Problem size sweep** – confirm the saturation point where performance becomes memory‑bound.  
2. **Work‑per‑thread sweep** – compare an *efficient* (contiguous) and a *non‑efficient* (strided) mapping.  
3. **Compute‑intensity sweep** – increase arithmetic intensity by adding extra fused‑multiply‑adds inside the kernel and map the results onto a Roofline plot.

👉 **Prerequisites**  
* You have built a command‑line executable called `lab_proj` (or adjust the path below) that accepts the following **positional** arguments:  
  `./lab_proj <width> <elements_per_thread> <loop_count> <access_mode>`  
  * `width` – total number of float elements (`N`).  
  * `elements_per_thread` – work items per thread (`W`).  
  * `loop_count` – inner loop iterations to raise compute intensity (`L`).  
  * `access_mode` – `0` → contiguous (efficient), `1` → strided (inefficient).  

  The program should print a single line of the form:  
  `KERNEL_TIME_MS=<milliseconds>`  
  The repository README explains how to adapt **lab_proj.cpp** and the kernel to accept these parameters.


In [ ]:

import subprocess, json, re, math, os, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXECUTABLE = "./lab_proj"

def run_kernel(n, w, l, mode, reps=3):
    """Run the executable several times and return average time in seconds."""
    times = []
    for _ in range(reps):
        completed = subprocess.run(
            [EXECUTABLE, str(n), str(w), str(l), str(mode)],
            capture_output=True, text=True, check=True
        )
        m = re.search(r"KERNEL_TIME_MS\s*=\s*([0-9.]+)", completed.stdout)
        if not m:
            raise RuntimeError(f"Failed to parse runtime from:\n{completed.stdout}")
        times.append(float(m.group(1)) / 1e3)  # → seconds
    return statistics.mean(times)

def bytes_transferred(n, w):
    """Total bytes moved per kernel call (2 reads + 1 write)."""
    return n * 3 * 4  # 4 bytes per float

def flops(n, w, l):
    """FLOPs per kernel: inner loop does 2 flops (mul+add)."""
    return n * l * 2


## 1. Problem‑size sweep – memory roof

In [ ]:

sizes = [2**k for k in range(16, 29)]  # 65 536 → 536 870 912
mem_bw = []
for n in sizes:
    t = run_kernel(n, 1, 1, 0)
    bw = bytes_transferred(n, 1) / t / 1e9  # GB/s
    mem_bw.append(bw)
    
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(sizes, mem_bw, marker='o')
plt.xscale('log')
plt.xlabel('Problem size N (floats)')
plt.ylabel('Effective memory bandwidth [GB/s]')
plt.title('Bandwidth vs. problem size')
plt.grid(True)
plt.show()


## 2. Work‑per‑thread sweep – mapping efficiency

In [ ]:

Ws = list(range(1, 33))  # 1 … 32 elements per thread
mode_labels = {0: "contiguous", 1: "strided"}
results = []
for mode in (0, 1):
    for w in Ws:
        n = 2**24  # keep global problem size fixed
        t = run_kernel(n, w, 1, mode)
        gflops = flops(n, w, 1) / t / 1e9
        results.append({"mode": mode, "W": w, "GFLOPS": gflops})
df_w = pd.DataFrame(results)

import seaborn as sns, matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
for mode, grp in df_w.groupby("mode"):
    plt.plot(grp["W"], grp["GFLOPS"], marker='o', label=mode_labels[mode])
plt.xlabel('elements per thread W')
plt.ylabel('Throughput [GFLOP/s]')
plt.title('Mapping strategy comparison')
plt.legend()
plt.grid(True)
plt.show()


## 3. Compute‑intensity sweep & Roofline

In [ ]:

Ls = [1, 2, 4, 8, 16, 32, 64, 128]
roof_data = []
peak_bw = max(mem_bw)             # GB/s observed
peak_flops_hw =  # <-- fill your GPU peak GFLOP/s
for L in Ls:
    n = 2**24
    t = run_kernel(n, 4, L, 0)
    gflops = flops(n, 4, L) / t / 1e9
    intensity = flops(n, 4, L) / bytes_transferred(n, 4)  # FLOP / byte
    roof_data.append((intensity, gflops))

# Plot Roofline
plt.figure(figsize=(6,4))
ispace = np.logspace(-2, 3, 200)
compute_roof = np.full_like(ispace, peak_flops_hw)
memory_roof = peak_bw * ispace
roof = np.minimum(compute_roof, memory_roof)
plt.loglog(ispace, roof, label='Roofline')
ints, gfs = zip(*roof_data)
plt.scatter(ints, gfs, marker='o', color='red', label='Measured')
for i,(x,y) in enumerate(roof_data):
    plt.text(x*1.1, y*1.1, f'L={Ls[i]}')
plt.xlabel('Arithmetic intensity [FLOP / Byte]')
plt.ylabel('Performance [GFLOP/s]')
plt.title('Roofline model vs. measurements')
plt.legend()
plt.grid(True, which='both')
plt.show()
